# 01 - Data and evidence channels

A short tour of the released DisAtlas inputs: the **broad** channel (shared genes and phenotypes), the **specific** channel (shared mechanism modules and mechanism nodes), the training graph built from their union, and pair scores from the five-seed ensemble.

Run from the repository root after fetching the Zenodo bundle into `data/zenodo` (see `data/README.md`). Only repository paths are read.

In [ ]:
import os
if not os.path.isdir('data') and os.path.isdir('../data'):
    os.chdir('..')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from disatlas.blinding import is_blinded
CH = 'data/raw/channels'
G = 'data/zenodo/graph'

## Channel tables

`broad_edges` are the disease-disease edges of the broad channel after thresholding. The disease-target tables keep Open Targets identifiers as released; targets from licence-restricted sources appear as stable salted tokens (`T_...`), with the source shown only at group level (`source_group`).

In [ ]:
broad = pd.read_parquet(f'{CH}/broad/broad_edges.parquet')
nodes = pd.read_parquet(f'{CH}/broad/broad_nodes.parquet')
dg = pd.read_parquet(f'{CH}/broad/disease_gene.parquet')
dp = pd.read_parquet(f'{CH}/broad/disease_phenotype.parquet')
links = pd.read_parquet(f'{CH}/specific/disease_module_links.parquet')
mech = pd.read_parquet(f'{CH}/specific/disease_target_mechanism.parquet')
union = pd.read_parquet(f'{G}/union_graph_edges.parquet')
train = pd.read_parquet(f'{G}/training_graph_edges.parquet')
def n_nodes(e): return pd.concat([e.u, e.v]).nunique()
pd.DataFrame({
    'edges': [len(broad), int(union.in_spec.sum()), len(union), len(train)],
    'nodes': [n_nodes(broad), n_nodes(union[union.in_spec]), n_nodes(union), n_nodes(train)],
}, index=['broad channel', 'specific channel', 'union graph', 'training graph (benchmark pairs removed)'])

In [ ]:
ov = union.assign(channel=np.select([union.in_broad & union.in_spec, union.in_broad], ['both', 'broad only'], 'specific only'))
ov.channel.value_counts().rename('edges').to_frame()

## Open Targets versus blinded sources

Share of disease-target rows by source group. In the broad table a gene can carry several sources (joined with `|`); `literature` and `curated_restricted` targets are blinded.

In [ ]:
def shares(d):
    blinded = d.target.map(is_blinded)
    return pd.Series({'rows': len(d), 'blinded share': blinded.mean(), 'any Open Targets source': d.source_group.str.contains('opentargets|OT_').mean()})
display(pd.DataFrame({'broad disease-gene': shares(dg), 'specific mechanism': shares(mech)}).T)
pd.concat({'broad disease-gene (top 10 source combinations)': dg.source_group.value_counts(normalize=True).head(10).round(3),
           'specific mechanism': mech.source_group.value_counts(normalize=True).round(3)}).to_frame('share')

## Degree distributions

In [ ]:
def degree(e): return pd.concat([e.u, e.v]).value_counts()
fig, ax = plt.subplots(figsize=(6, 3.5))
for name, e in [('broad', broad), ('specific', union[union.in_spec]), ('training graph', train)]:
    d = degree(e)
    ax.hist(d, bins=np.logspace(0, np.log10(d.max()), 40), histtype='step', label=f'{name} (median {int(d.median())})')
ax.set(xscale='log', yscale='log', xlabel='Disease degree', ylabel='Diseases')
ax.legend(frameon=False); fig.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(train.w_arm, bins=60)
ax.set(yscale='log', xlabel='Walk weight (w_arm)', ylabel='Edges'); fig.tight_layout()

## Ensemble pair scores

The released ensemble scores a disease pair as the mean over five seeds of the decoder probability. Below: the top-scoring partners of a few example diseases, and scores for some example pairs.

In [ ]:
from disatlas.model import load_ensemble
m = load_ensemble()
examples = ['MONDO:0005148', 'MONDO:0004975', 'MONDO:0007915']  # type 2 diabetes, Alzheimer disease, SLE
examples = [x for x in examples if x in set(m.ids)]
pd.concat({d: m.top_partners(d, k=5) for d in examples})

In [ ]:
pairs = pd.DataFrame({'disease_a': ['MONDO:0005148', 'MONDO:0007915', 'MONDO:0005148'],
                      'disease_b': ['MONDO:0005015', 'MONDO:0008383', 'MONDO:0004975']})
pairs['p_ensemble'] = m.score_pairs(pairs.disease_a, pairs.disease_b)
pairs